# 01.3 — Tokenization

**Problem it solves:** Models don't read sentences — they read sequences of tokens. Tokenization is the decision: what is the atomic unit of meaning?

**Why it matters:** Bad tokenization silently destroys information. "don't" split as ["don", "'", "t"] vs ["don't"] changes downstream meaning. The tokenizer is the first irreversible decision in your pipeline.

---

## Part 1: Whitespace and Punctuation Tokenizers

In [ ]:
# Level 0: split on whitespace
def whitespace_tokenize(text):
    return text.split()

# Level 1: split on whitespace AND punctuation
import re
def punct_tokenize(text):
    # Split on non-alphanumeric, keep the splitter
    return [t for t in re.split(r'(\W+)', text) if t.strip()]

test = "Hello, world! I can't believe it's 2024. Dr. Smith earned $100."
print("Whitespace:", whitespace_tokenize(test))
print()
print("Punct split:", punct_tokenize(test))

In [ ]:
# Level 2: proper word tokenizer
# Strategy: define what a TOKEN looks like, find all occurrences

WORD_TOKENIZE_PATTERN = re.compile(r"""
    (?:https?://\S+)          # URLs
    |(?:[A-Z]\.)+[A-Z]\.?    # Abbreviations: U.S.A.
    |\w+(?:-\w+)+             # Hyphenated: well-known
    |(?:\w+\'\w+)             # Contractions: can't, I'll
    |\w+                      # Regular words
    |[^\w\s]                  # Punctuation as single tokens
""", re.VERBOSE)

def word_tokenize(text):
    return WORD_TOKENIZE_PATTERN.findall(text)

tests = [
    "I can't believe it's happening!",
    "Dr. Smith visited the U.S.A. yesterday.",
    "The state-of-the-art model costs $2,500.",
    "Visit https://example.com for details.",
]
for t in tests:
    print(f"Input:  {t}")
    print(f"Tokens: {word_tokenize(t)}")
    print()

## Part 2: Character Tokenizer

Used in character-level language models — predicts one character at a time.

In [ ]:
def char_tokenize(text):
    return list(text)

class CharVocab:
    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
    
    def build(self, corpus: str):
        chars = sorted(set(corpus))
        self.char2idx = {c: i for i, c in enumerate(chars)}
        self.idx2char = {i: c for c, i in self.char2idx.items()}
        return self
    
    def encode(self, text: str) -> list:
        return [self.char2idx[c] for c in text if c in self.char2idx]
    
    def decode(self, ids: list) -> str:
        return ''.join(self.idx2char[i] for i in ids)
    
    def __len__(self):
        return len(self.char2idx)

corpus = "hello world, this is a test. the quick brown fox."
vocab = CharVocab().build(corpus)
print(f"Vocab size: {len(vocab)}")
print(f"Vocab: {sorted(vocab.char2idx.keys())}")
encoded = vocab.encode("hello")
print(f"encode('hello') = {encoded}")
print(f"decode({encoded}) = {vocab.decode(encoded)!r}")

## Part 3: Build a Word-Level Vocabulary with UNK handling

In [ ]:
from collections import Counter

class WordVocab:
    PAD = '<PAD>'
    UNK = '<UNK>'
    BOS = '<BOS>'
    EOS = '<EOS>'
    
    def __init__(self, min_freq: int = 1, max_size: int = None):
        self.min_freq = min_freq
        self.max_size = max_size
        self.word2idx = {}
        self.idx2word = {}
        self.freqs = Counter()
    
    def build(self, sentences: list[list[str]]):
        for sent in sentences:
            self.freqs.update(sent)
        
        # Special tokens first
        specials = [self.PAD, self.UNK, self.BOS, self.EOS]
        vocab = specials[:]
        
        # Add words meeting frequency threshold, sorted by frequency
        candidates = [(w, f) for w, f in self.freqs.most_common() 
                      if f >= self.min_freq]
        if self.max_size:
            candidates = candidates[:self.max_size - len(specials)]
        
        vocab.extend(w for w, _ in candidates)
        
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        return self
    
    def encode(self, tokens: list[str]) -> list[int]:
        unk_idx = self.word2idx[self.UNK]
        return [self.word2idx.get(t, unk_idx) for t in tokens]
    
    def decode(self, ids: list[int]) -> list[str]:
        return [self.idx2word[i] for i in ids]
    
    def coverage(self, tokens: list[str]) -> float:
        known = sum(1 for t in tokens if t in self.word2idx)
        return known / len(tokens) if tokens else 0.0
    
    def __len__(self):
        return len(self.word2idx)

# Build on small corpus
corpus_sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "ran", "in", "the", "park"],
    ["a", "cat", "and", "a", "dog", "play"],
]
vocab = WordVocab(min_freq=1).build(corpus_sentences)
print(f"Vocab size: {len(vocab)}")
print(f"word2idx: {vocab.word2idx}")

test_tokens = ["the", "cat", "chased", "a", "xenomorph"]  # last two OOV
encoded = vocab.encode(test_tokens)
print(f"\nEncode {test_tokens}")
print(f"     = {encoded}")
print(f"Decode = {vocab.decode(encoded)}")
print(f"Coverage: {vocab.coverage(test_tokens):.0%}")

## Part 4: The OOV Problem and Why It Kills Word-Level Models

In [ ]:
# Word-level vocab on real data: the OOV problem
# Even with 50k words, you'll see OOV on:
# - proper nouns (people, places, products)
# - morphological variants (running/runs/ran -> same stem, different form)
# - typos, slang, neologisms
# - domain-specific terms

import random

# Simulate vocabulary coverage vs size
def estimate_coverage(word_freqs: Counter, vocab_sizes: list) -> dict:
    total_tokens = sum(word_freqs.values())
    results = {}
    for size in vocab_sizes:
        top_words = set(w for w, _ in word_freqs.most_common(size))
        covered = sum(f for w, f in word_freqs.items() if w in top_words)
        results[size] = covered / total_tokens
    return results

# Simulate a Zipf-distributed corpus (real language follows Zipf's law)
# Zipf: frequency ∝ 1/rank
n_unique = 100000
freqs = Counter({f'word_{i}': max(1, int(100000 / (i + 1))) 
                 for i in range(n_unique)})

coverage = estimate_coverage(freqs, [1000, 5000, 10000, 30000, 50000])
print("Vocabulary coverage by size (Zipf-distributed corpus):")
for size, cov in coverage.items():
    print(f"  {size:6,} words -> {cov:.1%} token coverage")

print("\nConclusion: word-level vocab hits diminishing returns quickly.")
print("This is why BPE (next notebook) was invented.")

## Summary

**What it does:** Converts raw text into a sequence of discrete units a model can process.

**When to use which:**
- Character-level: language modeling, morphology, spell correction
- Word-level: when vocab is controlled and OOV is rare (closed-domain)
- Subword (BPE, next notebook): the modern default — handles OOV gracefully

**What breaks it:**
- Word tokenizers fragment morphologically rich languages (Turkish, Finnish)
- Contractions, hyphenation, and clitics vary by language
- Whitespace doesn't delimit words in Chinese/Japanese/Thai
- Any tokenizer that uses fixed vocab will have an OOV problem

---
**Next:** 01.4 — Byte Pair Encoding: the tokenizer used by every modern LLM